# Taxi Trip Duration Prediction


In [ ]:
import matplotlib.pyplot as plt
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import (
    OneHotEncoder,
    StandardScaler,
    StringIndexer,
    VectorAssembler,
)
from pyspark.ml.regression import GBTRegressor, LinearRegression, RandomForestRegressor
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, unix_timestamp

In [ ]:
spark = (
    SparkSession.builder.config("spark.executor.memory", "10g")
    .config("spark.driver.memory", "10g")
    .appName("trip_duration_prediction")
    .getOrCreate()
)
spark

In [ ]:
df = spark.read.parquet("datasets/yellow_tripdata_2015-01.parquet")
df.show()

In [ ]:
df.printSchema()

In [ ]:
df = df.na.drop()
df.count()

## Data Cleaning


Clean invalid data


In [ ]:
# New York is bounded by the coordinates (latitude, longitude) = (40.5774, -74.15) and (40.9176, -73.7004)

df = df.where(
    df.pickup_longitude.between(-74.15, -73.7004)
    & df.pickup_latitude.between(40.5774, 40.9176)
    & df.dropoff_longitude.between(-74.15, -73.7004)
    & df.dropoff_latitude.between(40.5774, 40.9176)
)
df.count()

In [ ]:
df = df.withColumn(
    "trip_duration",
    (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime"))
    / 60,
)
df.show()

In [ ]:
# As per NYC TLC Regulations, the maximum allowed trip time in a 24 hour interval is 12 hours.

df = df.where(df.trip_duration.between(0, 12 * 60))
df.select("trip_duration").orderBy("trip_duration", ascending=False).show()
df.count()

In [ ]:
df = df.drop("tpep_pickup_datetime", "tpep_dropoff_datetime")
df

In [ ]:
# NYC Taxi can hold up to 6 passengers

df = df.where(df.passenger_count.between(0, 6))
df.count()

In [ ]:
# Feature only includes $0.50 and $1 rush hour and overnight charges.

df = df.where(df.extra.between(0, 1.5))
df.count()

In [ ]:
# 0.5 MTA tax is automatically triggered.

df = df.where(col("mta_tax") == 0.5)
df.count()

In [ ]:
# 0.30 improvement surcharge, began being levied in 2015

df = df.where(
    (col("improvement_surcharge") == 0.3) | (col("improvement_surcharge") == 0.0)
)
df.count()

Clean data based on analysis


In [ ]:
df = df.where(df.trip_distance.between(0, 302.1))
df = df.where(df.fare_amount.between(0, 441.6))
df = df.where(df.tip_amount.between(0, 888.2))
df = df.where(df.tolls_amount.between(0, 33.0436))
df = df.where(df.total_amount.between(-400.3, 4009.3))
df.count()

In [ ]:
df.describe(
    [
        "passenger_count",
        "trip_distance",
        "pickup_longitude",
        "pickup_latitude",
        "dropoff_longitude",
        "dropoff_latitude",
        "fare_amount",
        "extra",
        "mta_tax",
        "tip_amount",
        "tolls_amount",
        "improvement_surcharge",
        "total_amount",
        "trip_duration",
    ]
).show()

In [ ]:
df.plot.hist(column="trip_duration", title="trip_duration").show()
df.plot.box(column="trip_duration").show()

## Data Preprocessing


In [ ]:
train, test = df.randomSplit([0.9, 0.1], seed=42)
train.cache()
test.cache()

In [ ]:
indexer = StringIndexer(
    inputCols=["store_and_fwd_flag", "RateCodeID"],
    outputCols=["store_and_fwd_flag_indexed", "RateCodeID_indexed"],
)

encoder = OneHotEncoder(
    inputCols=[
        "VendorID",
        "RateCodeID_indexed",
        "store_and_fwd_flag_indexed",
        "payment_type",
    ],
    outputCols=[
        "VendorID_vec",
        "RateCodeID_vec",
        "store_and_fwd_flag_vec",
        "payment_type_vec",
    ],
    handleInvalid="keep",
)

In [ ]:
assembler = VectorAssembler(
    inputCols=[
        "passenger_count",
        "trip_distance",
        "pickup_longitude",
        "pickup_latitude",
        "dropoff_longitude",
        "dropoff_latitude",
        "fare_amount",
        "extra",
        "mta_tax",
        "tip_amount",
        "tolls_amount",
        "improvement_surcharge",
        "total_amount",
    ]
    + [
        "VendorID_vec",
        "RateCodeID_vec",
        "store_and_fwd_flag_vec",
        "payment_type_vec",
    ],
    outputCol="feature_vector",
)

scaler = StandardScaler(inputCol="feature_vector", outputCol="scaled_feature_vector")

# Prediction


Linear Regression


In [ ]:
linear_regression = LinearRegression(
    featuresCol="feature_vector", labelCol="trip_duration"
)

pipeline = Pipeline(stages=[indexer, encoder, assembler, scaler, linear_regression])

evaluator = RegressionEvaluator(predictionCol="prediction", labelCol="trip_duration")

In [ ]:
param_grid = (
    ParamGridBuilder().addGrid(linear_regression.regParam, [0.01, 0.1, 1.0]).build()
)

train_validation = TrainValidationSplit(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    trainRatio=0.9,
    seed=42,
)

In [ ]:
model = train_validation.fit(train)
predictions = model.transform(test)
predictions.show()

In [ ]:
best_model = model.bestModel.stages[-1]
print("regParam", best_model.getRegParam())

In [ ]:
linear_regression_rmse = evaluator.evaluate(predictions)
linear_regression_rmse

Random Forest


In [ ]:
random_forest = RandomForestRegressor(
    featuresCol="feature_vector", labelCol="trip_duration", seed=42
)

pipeline = Pipeline(stages=[indexer, encoder, assembler, scaler, random_forest])

In [ ]:
param_grid = (
    ParamGridBuilder()
    .addGrid(random_forest.minInstancesPerNode, [1, 5])
    .addGrid(random_forest.minInfoGain, [0, 0.1])
    .build()
)

train_validation = TrainValidationSplit(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    trainRatio=0.9,
    seed=42,
)

In [ ]:
model = train_validation.fit(train)
predictions = model.transform(test)
predictions.show()

In [ ]:
best_model = model.bestModel.stages[-1]
print("minInstancesPerNode", best_model.getMinInstancesPerNode())
print("minInfoGain", best_model.getMinInfoGain())

In [ ]:
random_forest_rmse = evaluator.evaluate(predictions)
random_forest_rmse

Gradient-Boosted Trees


In [ ]:
gradient_boosted_trees = GBTRegressor(
    featuresCol="feature_vector", labelCol="trip_duration", seed=42
)

pipeline = Pipeline(
    stages=[indexer, encoder, assembler, scaler, gradient_boosted_trees]
)

In [ ]:
param_grid = (
    ParamGridBuilder()
    .addGrid(gradient_boosted_trees.minInstancesPerNode, [1, 5])
    .addGrid(gradient_boosted_trees.stepSize, [0.01, 0.1])
    .build()
)

train_validation = TrainValidationSplit(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    trainRatio=0.9,
    seed=42,
)

In [ ]:
model = train_validation.fit(train)
predictions = model.transform(test)
predictions.show()

In [ ]:
best_model = model.bestModel.stages[-1]
print("minInstancesPerNode", best_model.getMinInstancesPerNode())
print("stepSize", best_model.getStepSize())

In [ ]:
gradient_boosted_trees_rmse = evaluator.evaluate(predictions)
gradient_boosted_trees_rmse

Model Comparison


In [ ]:
plt.bar(
    ["Linear Regression", "Random Forest", "Gradient-Boosted Trees"],
    [linear_regression_rmse, random_forest_rmse, gradient_boosted_trees_rmse],
)
plt.title("RMSE")
plt.show()